# 06 - SHAP Explainability
Global + individual SHAP explanations for the winning model.
Model loaded dynamically from winner_meta.csv (never hardcoded to a specific algorithm).
Narratives generated for all PRIORITY companies.
**Input:** winner_calibrated.pkl, winner_meta.csv, prospective_final.csv

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import shap
from src.config import NARRATIVES_DIR, PROCESSED_FILES, MODELS_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR
print(f"Imports OK | SHAP version: {shap.__version__}")

Imports OK | SHAP version: 0.51.0


In [2]:
# nace_notes: Sector descriptions for enhanced SHAP narratives
# nace_notes contains plain-English descriptions of each NACE sector (e.g. "F = Construction")
# Used here to make SHAP narratives readable for EY audit teams

import pandas as pd
from pathlib import Path
from src.config import RAW_FILES

# Standard NACE section labels (A-S) - always populated as the safe baseline
# so narratives always have plain-English sector descriptions even if the
# enrichment file is missing or unreadable.
SECTOR_LABELS = {
    "A": "Agriculture, Forestry and Fishing",
    "B": "Mining and Quarrying",
    "C": "Manufacturing",
    "D": "Electricity, Gas and Steam Supply",
    "E": "Water Supply and Waste Management",
    "F": "Construction",
    "G": "Wholesale and Retail Trade",
    "H": "Transportation and Storage",
    "I": "Accommodation and Food Service",
    "J": "Information and Communication",
    "K": "Financial and Insurance Activities",
    "L": "Real Estate Activities",
    "M": "Professional and Scientific Activities",
    "N": "Administrative and Support Services",
    "O": "Public Administration and Defence",
    "P": "Education",
    "Q": "Human Health and Social Work",
    "R": "Arts, Entertainment and Recreation",
    "S": "Other Service Activities",
}

# If a nace_notes file is registered and present, extend SECTOR_LABELS with
# any richer subdivision-level labels it provides. Dispatches on file
# extension so both .xlsx and .csv are handled.
try:
    nace_notes_path = RAW_FILES.get("nace_notes", None)
    if nace_notes_path and Path(nace_notes_path).exists():
        suffix = Path(nace_notes_path).suffix.lower()
        if suffix in (".xlsx", ".xls"):
            notes_df = pd.read_excel(nace_notes_path)
        else:
            # Try several text encodings before giving up
            notes_df = None
            for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
                try:
                    notes_df = pd.read_csv(nace_notes_path, encoding=enc)
                    break
                except UnicodeDecodeError:
                    continue
            if notes_df is None:
                raise ValueError("Could not decode nace_notes with any known encoding")

        notes_df.columns = [c.strip().lower().replace(" ", "_") for c in notes_df.columns]
        sec_col  = next((c for c in notes_df.columns if "section" in c or "code" in c), notes_df.columns[0])
        desc_col = next((c for c in notes_df.columns if "desc" in c or "name" in c or "label" in c),
                        notes_df.columns[1] if len(notes_df.columns) > 1 else notes_df.columns[0])

        added = 0
        for _, row in notes_df.iterrows():
            code = str(row[sec_col]).strip().upper()
            desc = str(row[desc_col]).strip()
            if code and desc and code != "NAN" and desc != "NAN" and len(code) <= 4:
                SECTOR_LABELS[code] = desc
                added += 1
        print(f"nace_notes loaded ({suffix}) - {added} subdivision labels added; {len(SECTOR_LABELS)} total")
    else:
        print(f"nace_notes file not found - using {len(SECTOR_LABELS)} hardcoded NACE section labels")
except Exception as e:
    print(f"nace_notes WARNING: {e} - falling back to {len(SECTOR_LABELS)} hardcoded NACE section labels")

def get_sector_label(nace_code):
    """Return plain-English NACE sector label for a NACE code (e.g. F41 -> Construction)"""
    if pd.isna(nace_code) or str(nace_code).strip() == "":
        return "Unclassified"
    code = str(nace_code).strip().upper()
    # Try direct match first (handles subdivision codes from enriched file)
    if code in SECTOR_LABELS:
        return SECTOR_LABELS[code]
    # Then section letter
    if code[0] in SECTOR_LABELS:
        return SECTOR_LABELS[code[0]]
    # Try 2-digit division lookup
    div = "".join(c for c in code if c.isdigit())[:2]
    div_to_sec = {}
    for d in range(1, 4):  div_to_sec[str(d).zfill(2)] = "A"
    for d in range(5, 10): div_to_sec[str(d).zfill(2)] = "B"
    for d in range(10, 34): div_to_sec[str(d)] = "C"
    div_to_sec.update({"35": "D"})
    for d in range(36, 40): div_to_sec[str(d)] = "E"
    for d in range(41, 44): div_to_sec[str(d)] = "F"
    for d in range(45, 48): div_to_sec[str(d)] = "G"
    for d in range(49, 54): div_to_sec[str(d)] = "H"
    for d in range(55, 57): div_to_sec[str(d)] = "I"
    for d in range(58, 64): div_to_sec[str(d)] = "J"
    for d in range(64, 67): div_to_sec[str(d)] = "K"
    div_to_sec.update({"68": "L"})
    for d in range(69, 76): div_to_sec[str(d)] = "M"
    for d in range(77, 83): div_to_sec[str(d)] = "N"
    sec = div_to_sec.get(div, None)
    if sec: return SECTOR_LABELS.get(sec, f"Sector {code}")
    return f"Sector {code}"

print(f"\nExample NACE label: F41 -> {get_sector_label('F41')}")
print(f"Example NACE label: 6499 -> {get_sector_label('6499')}")
print(f"nace_notes - sector labels ready for SHAP narrative generation")

nace_notes loaded (.xlsx) - 396 subdivision labels added; 396 total

Example NACE label: F41 -> CONSTRUCTION
Example NACE label: 6499 -> TELECOMMUNICATION, COMPUTER PROGRAMMING, CONSULTING, COMPUTING INFRASTRUCTURE AND OTHER INFORMATION SERVICE ACTIVITIES
nace_notes - sector labels ready for SHAP narrative generation


In [3]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd, numpy as np
from src.config import PROCESSED_FILES, MODELS_DIR, OUTPUTS_DIR, FIGURES_DIR

_meta_df = pd.read_csv(MODELS_DIR / "winner_meta.csv")
meta     = dict(zip(_meta_df["key"], _meta_df["value"]))
winner_name = meta["winner_name"]
with open(MODELS_DIR / "feature_cols.txt") as _f:
    FEATURE_COLS_M = [l.strip() for l in _f if l.strip()]

import joblib as _jl
fname      = winner_name.lower().replace(" ", "_") + "_model.joblib"
winner_model = _jl.load(MODELS_DIR / fname)

train_df  = pd.read_csv(PROCESSED_FILES["train_set"], low_memory=False)
test_df   = pd.read_csv(PROCESSED_FILES["test_set"],  low_memory=False)
PROSP_FINAL = PROCESSED_FILES["prospective_set"].parent / "prospective_final.csv"
prosp_df  = pd.read_csv(PROSP_FINAL, low_memory=False)

X_test = test_df[FEATURE_COLS_M].values
y_test = test_df["label"].values.astype(int)

print(f"Winner model  : {winner_name}")
print(f"Features      : {len(FEATURE_COLS_M)}")
print(f"Test rows     : {len(X_test):,}")
print(f"Prospective   : {len(prosp_df):,}")

Winner model  : XGBoost
Features      : 84
Test rows     : 94,421
Prospective   : 28,974


In [4]:
# Compute SHAP values
rng = np.random.default_rng(42)
act_idx = np.where(y_test==0)[0]; dis_idx = np.where(y_test==1)[0]
n_act = min(2000, len(act_idx));  n_dis = min(500, len(dis_idx))
idx_s = np.concatenate([rng.choice(act_idx, n_act, replace=False),
                        rng.choice(dis_idx, n_dis, replace=False)])
X_sample = X_test[idx_s]; y_sample = y_test[idx_s]

print(f"SHAP sample: {len(X_sample):,} rows  ({n_act} active + {n_dis} distress)")
print(f"Computing SHAP values for {winner_name}...")

TREE_MODELS = ("LightGBM", "XGBoost", "Random Forest", "Decision Tree")

if winner_name in TREE_MODELS:
    explainer  = shap.TreeExplainer(winner_model)
    shap_raw   = explainer.shap_values(X_sample, check_additivity=False)
    if isinstance(shap_raw, list):
        shap_vals = shap_raw[1]; base_val  = explainer.expected_value[1]
    else:
        shap_vals = shap_raw
        base_val  = (explainer.expected_value[-1]
                     if isinstance(explainer.expected_value, (list, np.ndarray))
                     else explainer.expected_value)
else:
    X_sample_scaled = winner_model[:-1].transform(X_sample)
    lr_clf          = winner_model[-1]
    explainer  = shap.LinearExplainer(lr_clf, X_sample_scaled)
    shap_raw   = explainer.shap_values(X_sample_scaled)
    shap_vals  = shap_raw[1] if isinstance(shap_raw, list) else shap_raw
    base_val   = (explainer.expected_value[1]
                  if isinstance(explainer.expected_value, (list, np.ndarray))
                  else explainer.expected_value)
    X_sample_for_plots = X_sample_scaled

def sigmoid(x): return 1 / (1 + np.exp(-x))

print(f"SHAP values shape : {shap_vals.shape}")
print(f"Base value (internal): {base_val:.4f}  sigmoid={sigmoid(base_val)*100:.2f}% (weighted model boundary)")
actual_base_rate = train_df["label"].mean()
print(f"Actual training dissolution rate: {actual_base_rate:.2%}  <- use this in narratives")
np.save(OUTPUTS_DIR / "step6_shap_values_test.npy", shap_vals)
print("SHAP values computed and saved")

SHAP sample: 2,500 rows  (2000 active + 500 distress)
Computing SHAP values for XGBoost...
SHAP values shape : (2500, 84)
Base value (internal): 0.0964  sigmoid=52.41% (weighted model boundary)
Actual training dissolution rate: 6.69%  <- use this in narratives
SHAP values computed and saved


In [5]:
# Global importance bar chart
mean_abs    = np.abs(shap_vals).mean(axis=0)
sorted_idx  = np.argsort(mean_abs)
sorted_names = [FEATURE_COLS_M[i] for i in sorted_idx]
sorted_vals  = mean_abs[sorted_idx]

fig, ax = plt.subplots(figsize=(9, 8))
bars = ax.barh(range(len(sorted_names)), sorted_vals, color="#E91E63", alpha=0.85, edgecolor="white")
ax.set_yticks(range(len(sorted_names))); ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlabel("Mean |SHAP values| (log-odds)", fontsize=10)
ax.set_title(f"Global Feature Importance: All {len(FEATURE_COLS_M)} Features\n{winner_name} - Irish SME Dissolution Risk",
             fontsize=11, fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
for bar, val in zip(bars, sorted_vals):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_shap_bar.png", dpi=150, bbox_inches="tight"); plt.close()

plt.figure(figsize=(9, 7))
shap.summary_plot(shap_vals, X_sample, feature_names=FEATURE_COLS_M,
                  show=False, plot_size=None, max_display=len(FEATURE_COLS_M))
plt.title(f"SHAP Summary - All {len(FEATURE_COLS_M)} Features\n{winner_name} (n={len(X_sample):,})",
          fontsize=11, fontweight="bold", pad=12)
plt.xlabel("SHAP value (positive = increases dissolution risk)", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.close()

print("GLOBAL IMPORTANCE RANKING (Mean |SHAP|)")
print("="*60)
for rank, (name, val) in enumerate(zip(sorted_names[::-1], sorted_vals[::-1]), 1):
    bar = "|" * int(val/max(sorted_vals)*25)
    print(f"  #{rank:<2} {name:<35}: {val:.4f}  {bar}")
print("\nSaved: step6_shap_bar.png, step6_shap_beeswarm.png")

GLOBAL IMPORTANCE RANKING (Mean |SHAP|)
  #1  ar_filed_count                     : 3.7304  |||||||||||||||||||||||||
  #2  company_age_years                  : 2.7894  ||||||||||||||||||
  #3  annual_submission_rate             : 2.6677  |||||||||||||||||
  #4  total_submissions                  : 2.6471  |||||||||||||||||
  #5  director_change_count              : 1.6581  |||||||||||
  #6  submission_history_years           : 1.0266  ||||||
  #7  other_form_count                   : 0.8730  |||||
  #8  age_vs_sector_median               : 0.7313  ||||
  #9  days_since_last_name_change        : 0.6710  ||||
  #10 name_change_count                  : 0.5075  |||
  #11 days_since_last_office_change      : 0.4587  |||
  #12 name_risk_score                    : 0.3899  ||
  #13 sector_relative_deviation          : 0.3506  ||
  #14 office_change_count                : 0.3483  ||
  #15 same_day_dissolution_count         : 0.3068  ||
  #16 same_day_reg_count                 : 0.2658  |
  #17 

In [6]:
# Dependence plots: top 3 features
top3_idx   = np.argsort(mean_abs)[::-1][:3]
top3_names = [FEATURE_COLS_M[i] for i in top3_idx]
fig, axes  = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"SHAP Dependence Plots - Top 3 Features ({winner_name})", fontsize=12, fontweight="bold")
colours = ["#E91E63","#FF9800","#2196F3"]
for ax, fi, fn, col in zip(axes, top3_idx, top3_names, colours):
    fv = X_sample[:, fi]; sv = shap_vals[:, fi]
    ax.scatter(fv[y_sample==0], sv[y_sample==0], alpha=0.15, s=5, c="#9E9E9E", label="Active")
    ax.scatter(fv[y_sample==1], sv[y_sample==1], alpha=0.5,  s=10, c=col,      label="Distress")
    ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
    ax.set_xlabel(fn, fontsize=10); ax.set_ylabel("SHAP value", fontsize=10)
    ax.set_title(fn, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_shap_dependence.png", dpi=150, bbox_inches="tight"); plt.close()
print("Saved: step6_shap_dependence.png")

Saved: step6_shap_dependence.png


In [7]:
# SHAP for PRIORITY companies
priority_df = (prosp_df[prosp_df["combined_risk_tier"]=="PRIORITY"]
               .sort_values("combined_risk_score", ascending=False)
               .reset_index(drop=True))
X_priority_raw = priority_df[FEATURE_COLS_M].values
print(f"Computing SHAP for {len(priority_df)} PRIORITY companies ({winner_name})...")
if winner_name in TREE_MODELS:
    X_priority = X_priority_raw
    prio_raw   = explainer.shap_values(X_priority, check_additivity=False)
else:
    X_priority = winner_model[:-1].transform(X_priority_raw)
    prio_raw   = explainer.shap_values(X_priority)
prio_shap  = prio_raw[1] if isinstance(prio_raw, list) else prio_raw

print("SANITY CHECK (sigmoid(base + SHAP_sum) vs dissolution_risk_score):")
for i in range(min(3, len(priority_df))):
    shap_p  = sigmoid(base_val + prio_shap[i].sum())
    model_p = priority_df.iloc[i]["dissolution_risk_score"]
    match   = "OK" if abs(shap_p - model_p) < 0.05 else "MISMATCH"
    print(f"  Company {priority_df.iloc[i]['company_num']}: SHAP={shap_p:.4f}  Score={model_p:.4f}  {match}")

Computing SHAP for 78 PRIORITY companies (XGBoost)...
SANITY CHECK (sigmoid(base + SHAP_sum) vs dissolution_risk_score):
  Company 653915: SHAP=0.9988  Score=0.9583  OK
  Company 632240: SHAP=0.9999  Score=0.9747  OK
  Company 707700: SHAP=1.0000  Score=1.0000  OK


In [8]:
# Per-company SHAP for the full prospective cohort (dashboard / presentation layer)
# Applies the same explainer used above to every prospective company and saves a
# compact per-company driver table. Deterministic; no model change. This is the
# lookup the company-level presentation reads, so every searched company has its
# own risk drivers rather than only the PRIORITY tier.

import json

X_prosp_raw = prosp_df[FEATURE_COLS_M].fillna(0).values
print(f"Computing SHAP for {len(prosp_df):,} prospective companies ({winner_name})...")

if winner_name in TREE_MODELS:
    X_prosp = X_prosp_raw
    prosp_raw_shap = explainer.shap_values(X_prosp, check_additivity=False)
else:
    X_prosp = winner_model[:-1].transform(X_prosp_raw)
    prosp_raw_shap = explainer.shap_values(X_prosp)
prosp_shap = prosp_raw_shap[1] if isinstance(prosp_raw_shap, list) else prosp_raw_shap

# Build a compact per-company record: full SHAP vector plus a ranked top-8 with
# feature value and direction, so the presentation layer needs no recomputation.
N_TOP = 8
records = []
feature_arr = np.array(FEATURE_COLS_M)
for i in range(len(prosp_df)):
    row = prosp_df.iloc[i]
    sv = prosp_shap[i]
    order = np.argsort(np.abs(sv))[::-1][:N_TOP]
    top = []
    for fi in order:
        fname = FEATURE_COLS_M[fi]
        top.append({
            "feature": fname,
            "value": float(row.get(fname, 0)),
            "shap": round(float(sv[fi]), 4),
            "direction": "increases" if sv[fi] > 0 else "decreases",
        })
    records.append({
        "company_num": row.get("company_num"),
        "company_name": row.get("company_name"),
        "combined_risk_tier": row.get("combined_risk_tier"),
        "dissolution_risk_score": row.get("dissolution_risk_score"),
        "base_value": round(float(base_val), 4),
        "shap_sum": round(float(sv.sum()), 4),
        "top_drivers_json": json.dumps(top),
    })

prosp_shap_df = pd.DataFrame(records)
shap_csv = OUTPUTS_DIR / "prospective_shap.csv"
prosp_shap_df.to_csv(shap_csv, index=False)

# Also save the raw matrix for any downstream numeric use.
np.save(OUTPUTS_DIR / "prospective_shap_values.npy", prosp_shap)

print(f"Saved per-company SHAP drivers : {shap_csv}  ({len(prosp_shap_df):,} rows)")
print(f"Saved raw SHAP matrix          : prospective_shap_values.npy  {prosp_shap.shape}")

# Coverage check by tier so it is clear every tier is populated.
print("\nPer-company SHAP coverage by tier:")
for t in ["PRIORITY", "DISSOLUTION_RISK", "BEHAVIORAL_ANOMALY", "LOW_CONCERN"]:
    n = int((prosp_shap_df["combined_risk_tier"] == t).sum())
    if n:
        print(f"  {t:<20} {n:>7,}")


Computing SHAP for 28,974 prospective companies (XGBoost)...
Saved per-company SHAP drivers : E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\outputs\prospective_shap.csv  (28,974 rows)
Saved raw SHAP matrix          : prospective_shap_values.npy  (28974, 84)

Per-company SHAP coverage by tier:
  PRIORITY                  78
  DISSOLUTION_RISK       5,833
  BEHAVIORAL_ANOMALY     1,578
  LOW_CONCERN           21,485


In [9]:
# Waterfall plots + narrative generation
def plot_waterfall(ax, sv, fv, fnames, bv, cnum, risk, anom):
    idx    = np.argsort(np.abs(sv))[::-1]
    s_v    = sv[idx]
    labels = [f"{fnames[i]}\n={fv[i]:.3f}" for i in idx]
    cols   = ["#E24B4A" if v > 0 else "#378ADD" for v in s_v]
    ax.barh(range(len(fnames)), s_v, color=cols, edgecolor="white", lw=0.5)
    ax.set_yticks(range(len(fnames))); ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("SHAP value (log-odds)", fontsize=8)
    ax.set_title(f"Company {cnum}\nRisk={risk:.3f}  Anomaly={anom:.3f}", fontsize=9, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.3)
    total = bv + sv.sum()
    ax.text(0.98, 0.02, f"base={bv:.2f}\ntotal={total:.2f}\np={sigmoid(total):.3f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=6,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.4))

n_plot = min(5, len(priority_df))
fig, axes = plt.subplots(1, n_plot, figsize=(4*n_plot, 9))
if n_plot == 1: axes = [axes]
fig.suptitle(f"SHAP Waterfall - Top {n_plot} PRIORITY Companies ({winner_name})", fontsize=12, fontweight="bold")
for i, ax in enumerate(axes):
    row = priority_df.iloc[i]
    plot_waterfall(ax, prio_shap[i], X_priority[i], FEATURE_COLS_M, base_val,
                   row["company_num"], row["dissolution_risk_score"],
                   row.get("if_anomaly_score", 0))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "step6_shap_waterfall_priority.png", dpi=150, bbox_inches="tight"); plt.close()
print("Saved: step6_shap_waterfall_priority.png")

Saved: step6_shap_waterfall_priority.png


In [10]:
# Generate EY-standard narratives
AUDIT_STEPS = [
    "Verify current director details and registered address on CRO website (services.cro.ie)",
    "Request most recent 3 years of management accounts from company",
    "Check for outstanding Revenue Commissioners or CRO compliance notices",
    "Contact registered address to confirm the company is still actively trading",
    "Review director network for cross-directorships with other distressed entities",
]

def generate_ey_narrative(row, sv, fnames, bv, actual_base_rate):
    risk_prob  = row.get("dissolution_risk_score", 0)
    anom_score = row.get("if_anomaly_score", 0)
    comb_score = row.get("combined_risk_score", 0)
    if risk_prob >= 0.30:
        action = "IMMEDIATE AUDIT TRIGGER - escalate to engagement partner within 48 hours"
    elif risk_prob >= 0.15:
        action = "ENHANCED MONITORING - schedule director interview within 30 days"
    else:
        action = "WATCH LIST - include in next quarterly risk review cycle"
    top_idx = sorted(range(len(sv)), key=lambda x: abs(sv[x]), reverse=True)[:5]
    lines = [
        "="*72, f"PRIORITY COMPANY - EY DISSOLUTION RISK NARRATIVE", "="*72,
        f"Company Number : {row.get('company_num', '-')}",
        f"Company Name   : {row.get('company_name', 'Unknown')}",
        f"County         : {row.get('county', '-')}",
        f"Type           : {row.get('company_type', '-')}",
        f"Age            : {row.get('company_age_years', 0):.1f} years",
        "",
        "RISK SCORES",
        f"  Dissolution Risk Score : {risk_prob:.4f}  ({risk_prob:.1%} estimated probability)",
        f"  Behavioural Anomaly    : {anom_score:.4f}",
        f"  Combined Score         : {comb_score:.4f}",
        f"  Population Base Rate   : {actual_base_rate:.2%}  (training set dissolution rate)",
        f"  Relative Risk          : {risk_prob/max(actual_base_rate, 0.001):.1f}x above base rate",
        "",
        f"RECOMMENDED ACTION: {action}", "",
        "KEY RISK DRIVERS (top 5 by model contribution):",
    ]
    for rank, fi in enumerate(top_idx, 1):
        fname = fnames[fi]
        val   = row.get(fname, 0)
        s     = sv[fi]
        direction = "^ RISK" if s > 0 else "v risk"
        lines.append(f"  {rank}. {fname} = {val}  ->  {direction}  (|SHAP|={abs(s):.4f})")
    lines += ["", "SUGGESTED AUDIT STEPS:"]
    for step_i, step in enumerate(AUDIT_STEPS, 1):
        lines.append(f"  {step_i}. {step}")
    lines += ["", "NOTE: SHAP values explain raw model log-odds. Dissolution risk score is",
              "isotonic-calibrated. sigmoid(base + SHAP_sum) approximates but does not",
              "exactly equal the reported score (by design).", "="*72]
    return "\n".join(lines)

narratives = []
for i in range(len(priority_df)):
    narratives.append(generate_ey_narrative(
        priority_df.iloc[i], prio_shap[i], FEATURE_COLS_M, base_val, actual_base_rate))

sep    = "\n" + "="*72 + "\n"
header = (f"EY DISSOLUTION RISK - PRIORITY AUDIT TRIGGERS\n"
          f"Irish SME Dissolution Risk Model | {winner_name}\n"
          f"Total PRIORITY: {len(priority_df)} | Calibrated probabilities\n"
          f"Base rate: {actual_base_rate:.2%} (training dissolution rate)\n")
narr_path = TABLES_DIR / "step6_shap_narratives.txt"
with open(narr_path, "w", encoding="utf-8") as f:
    f.write(header); f.write(sep); f.write(sep.join(narratives))

print(f"EY-standard narratives saved for {len(priority_df):,} PRIORITY companies")

EY-standard narratives saved for 78 PRIORITY companies


In [11]:
ranked = sorted(zip(FEATURE_COLS_M, mean_abs), key=lambda x: x[1], reverse=True)
print("STEP 6 COMPLETE - SHAP EXPLAINABILITY")
print("="*65)
print(f"  Winner model       : {winner_name}")
print(f"  Features explained : {len(FEATURE_COLS_M)}")
print(f"  SHAP sample        : {len(X_sample):,} test companies")
print(f"  Base value         : {base_val:.4f}  ({sigmoid(base_val)*100:.2f}% base rate)")
print()
print("GLOBAL IMPORTANCE RANKING (mean |SHAP|):")
for rank, (name, val) in enumerate(ranked, 1):
    bar = "|" * int(val/ranked[0][1]*20)
    print(f"  #{rank:<2} {name:<35}: {val:.4f}  {bar}")
print()
print("OUTPUTS:")
for fname in ["step6_shap_bar.png","step6_shap_beeswarm.png","step6_shap_dependence.png",
              "step6_shap_waterfall_priority.png","step6_shap_narratives.txt","step6_shap_values_test.npy"]:
    print(f"  outputs/{fname}")
print()
print("PIPELINE COMPLETE: Steps 00 to 06 finished.")

STEP 6 COMPLETE - SHAP EXPLAINABILITY
  Winner model       : XGBoost
  Features explained : 84
  SHAP sample        : 2,500 test companies
  Base value         : 0.0964  (52.41% base rate)

GLOBAL IMPORTANCE RANKING (mean |SHAP|):
  #1  ar_filed_count                     : 3.7304  ||||||||||||||||||||
  #2  company_age_years                  : 2.7894  ||||||||||||||
  #3  annual_submission_rate             : 2.6677  ||||||||||||||
  #4  total_submissions                  : 2.6471  ||||||||||||||
  #5  director_change_count              : 1.6581  ||||||||
  #6  submission_history_years           : 1.0266  |||||
  #7  other_form_count                   : 0.8730  ||||
  #8  age_vs_sector_median               : 0.7313  |||
  #9  days_since_last_name_change        : 0.6710  |||
  #10 name_change_count                  : 0.5075  ||
  #11 days_since_last_office_change      : 0.4587  ||
  #12 name_risk_score                    : 0.3899  ||
  #13 sector_relative_deviation          : 0.3506  |
 

## Parsimony analysis: top-15 SHAP features

The full model uses all features so that the financial-ablation argument can be
made on the complete feature set. This section reports how much predictive
performance a compact model retains when restricted to the fifteen
highest-ranked features by mean absolute SHAP value. A small drop in Average
Precision indicates that the signal is concentrated in a handful of behavioural
features, and that the remaining features contribute marginal refinement rather
than core discrimination.

In [12]:
# Retrain the winner on the top-15 SHAP features and report AP retention.
# The full model remains the primary model; this is a parsimony robustness result.
from sklearn.metrics import average_precision_score, roc_auc_score
import xgboost as _xgb

top15 = [name for name, _ in ranked[:15]]
with open(MODELS_DIR / "top15_features.txt", "w") as _f:
    _f.write("\n".join(top15))

y_tr = train_df["label"].values.astype(int)
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

def _fit_ap(features):
    Xtr = train_df[features].fillna(0).values
    Xte = test_df[features].fillna(0).values
    m = _xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8,
        eval_metric='aucpr', random_state=42, n_jobs=-1)
    m.fit(Xtr, y_tr)
    p = m.predict_proba(Xte)[:, 1]
    return average_precision_score(y_test, p), roc_auc_score(y_test, p)

ap_full, auc_full = _fit_ap(FEATURE_COLS_M)
ap_top15, auc_top15 = _fit_ap(top15)
retention = 100 * ap_top15 / ap_full if ap_full else 0

print("PARSIMONY ANALYSIS - top 15 SHAP features")
print("=" * 60)
print(f"  Full model ({len(FEATURE_COLS_M)} feat): AP={ap_full:.4f}  AUC={auc_full:.4f}")
print(f"  Top-15 model         : AP={ap_top15:.4f}  AUC={auc_top15:.4f}")
print(f"  AP retention         : {retention:.1f}% of full-model AP")
print()
print("  Top 15 features:")
for _i, _f in enumerate(top15, 1):
    print(f"    {_i:>2}. {_f}")

import pandas as _pd
_pd.DataFrame({"rank": range(1, 16), "feature": top15,
    "full_ap": round(ap_full, 4), "top15_ap": round(ap_top15, 4),
    "ap_retention_pct": round(retention, 1)}).to_csv(
    TABLES_DIR / "parsimony_top15.csv", index=False)
print(f"\nSaved: {TABLES_DIR / 'parsimony_top15.csv'}")


PARSIMONY ANALYSIS - top 15 SHAP features
  Full model (84 feat): AP=0.5636  AUC=0.9341
  Top-15 model         : AP=0.5032  AUC=0.9180
  AP retention         : 89.3% of full-model AP

  Top 15 features:
     1. ar_filed_count
     2. company_age_years
     3. annual_submission_rate
     4. total_submissions
     5. director_change_count
     6. submission_history_years
     7. other_form_count
     8. age_vs_sector_median
     9. days_since_last_name_change
    10. name_change_count
    11. days_since_last_office_change
    12. name_risk_score
    13. sector_relative_deviation
    14. office_change_count
    15. same_day_dissolution_count

Saved: E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\outputs\tables\parsimony_top15.csv
